In [48]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import numpy as np
import networkx as nx 

# Load Data

In [50]:
# Route graph
G = nx.read_gml('../data/graph/LEMD_EGLL_2023_04_01.gml')
print(f'There are {len(G.nodes())} nodes in the graph, and {len(G.edges())} edges')

# Transitions
import pickle
with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_REACHABLE.pkl', 'rb') as f:
    transitions = pickle.load(f)
print(f'There are {len(transitions)} transitions in the transitions array')

# Convert transitions to numpy array
transitions = np.array(transitions)

# Load the forward V_soft
V_soft = np.load('../data/graph/V_soft/LEMD_EGLL_2023_04_01_CLB_V_FWD.npy')

# (u_idx, k_u_std_idx, rho_u_std_idx, round(alt_u_std), phase_u_std,
# v_idx, k_v_idx, rho_v_idx, round(alt_v_val), phi_v_idx)

idx_to_node = {i: node for i, node in enumerate(G.nodes())}
node_to_idx = {node: i for i, node in enumerate(G.nodes())}

There are 566 nodes in the graph, and 5028 edges
There are 27939 transitions in the transitions array


In [51]:
def cost_proxy(distance_nm: float) -> float:
    return distance_nm * 1e-2

In [55]:
# Get topological sort of the nodes in G
try:
    topo_order = list(nx.topological_sort(G))
    print(f"Topological sort successful. First 10 nodes: {topo_order[:10]}")
    print(f"Total nodes in topological order: {len(topo_order)}")
except nx.NetworkXUnfeasible:
    print("Error: The graph G is not a Directed Acyclic Graph (DAG)")
except Exception as e:
    print(f"Error during topological sort: {e}")

lemd_successors = list(G.successors('LEMD'))
print(f'LEMD has {len(lemd_successors)} successors: {lemd_successors}')

print('='*20)

node_name_to_check = 'LETP'
id_of_node_to_check = node_to_idx[node_name_to_check]
print(f'Checking node: {node_name_to_check} (id: {id_of_node_to_check})')

# Find all predecessors of the chosen node
node_predecessors = list(G.predecessors(node_name_to_check))
print(f'Node {node_name_to_check} has {len(node_predecessors)} predecessors: {node_predecessors}')

# Find all preceding states
preceding_states = []
for predecessor in node_predecessors:
    # Calculate distance between predecessor and node_name_to_check
    pred_lat = G.nodes[predecessor]['lat']
    pred_lon = G.nodes[predecessor]['lon']
    node_lat = G.nodes[node_name_to_check]['lat']
    node_lon = G.nodes[node_name_to_check]['lon']
    
    # Haversine distance calculation
    from math import radians, cos, sin, asin, sqrt
    
    def haversine(lon1, lat1, lon2, lat2):
        """
        Calculate the great circle distance between two points 
        on the earth (specified in decimal degrees)
        Returns distance in nautical miles
        """
        # convert decimal degrees to radians 
        lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

        # haversine formula 
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * asin(sqrt(a)) 
        r = 3440.065  # Radius of earth in nautical miles
        return c * r
    
    distance_nm = haversine(pred_lon, pred_lat, node_lon, node_lat)
    print(f'Distance from {predecessor} to {node_name_to_check}: {distance_nm:.2f} nautical miles')
    print(f'Proxy cost between {predecessor} and {node_name_to_check}: {cost_proxy(distance_nm):.6f}')

    # Get the cost of the edge between predecessor and node_name_to_check

    # Get all the predecessor states of the chosen node
    print(f'='*20)
    predecessor_states = V_soft[node_to_idx[predecessor]]
    
    # Get eligible transitions first
    eligible_transitions = transitions[(transitions[:, 5] == id_of_node_to_check) & (transitions[:, 0] == node_to_idx[predecessor])]
    
    if len(eligible_transitions) > 0:
        print(predecessor)
        print(f"{'From State':<25} {'From Value':<12} {'To State':<25} {'To Value':<12}")
        print(f"{'Wall/Clb/Phase':<25} {'':<12} {'Wall/Clb/Phase':<25} {'':<12}")
        print("-" * 75)
        
        for transition in eligible_transitions:
            # From state (u)
            u_wall_clk = transition[1]
            u_clb_allwce = transition[2]
            u_phase = transition[4]
            u_value = predecessor_states[u_wall_clk, u_clb_allwce, u_phase]
            
            # To state (v)
            v_wall_clk = transition[6]
            v_clb_allwce = transition[7]
            v_phase = transition[9]
            v_value = V_soft[id_of_node_to_check, v_wall_clk, v_clb_allwce, v_phase]
            
            from_state_str = f"{u_wall_clk}/{u_clb_allwce}/{u_phase}"
            to_state_str = f"{v_wall_clk}/{v_clb_allwce}/{v_phase}"
            
            print(f"{from_state_str:<25} {u_value:<12.4f} {to_state_str:<25} {v_value:<12.4f}")
    else:
        print(f"No eligible transitions found for predecessor {predecessor}")
    
    


Topological sort successful. First 10 nodes: ['LEMD', 'DISKO_38', 'LETO', 'ZANKO', 'AVILA', 'CJN', 'RBO_16', 'LECV', 'DISKO', 'UNSOL_45']
Total nodes in topological order: 566
LEMD has 67 successors: ['RBO', 'RBO_16', 'LECV', 'LERM', 'LETP', 'PINAR', 'EDIGO_39', 'OSTIX', 'GASMO', 'SIE_06', 'SIE', 'UNSOL_45', 'DISKO_38', 'SEGRE', 'LETO', 'BASIM_84', 'ZANKO', 'AVILA', 'TITAN_91', 'BRITO_57', 'SEGRE_09', 'ZANKO_46', 'BAN_31', 'KONKE', 'EDIGO', 'YAKXU', 'BASIM', 'XERMA_10', 'DISKO', 'CJN', 'ORBIS_70', 'RATAS_86', 'JOCRE', 'GOSVI_78', 'LPA_13', 'VADOX_66', 'CEGAM_79', 'GOSVI', 'XORNA_80', 'PODUX', 'ROLES', 'BAKUP', 'BUGIX_59', 'BISKA_37', 'LFDA', 'BANEV', 'LEVD', 'NEA', 'SUSOS_49', 'SOVOS_00', 'ARLUN_82', 'ELSAP', 'SNR_12', 'AMTOS_09', 'NUBLO', 'UNGAS_63', 'NEDUS', 'AMTOS', 'MALOB_97', 'LFBP', 'BAN_14', 'ALEPO_09', 'CALCE', 'OSTIX_95', 'EMANU', 'BANEV_13', 'NONTU']
Checking node: LETP (id: 351)
Node LETP has 3 predecessors: ['LEMD', 'SIE', 'ORBIS_70']
Distance from LEMD to LETP: 43.97 nauti

In [62]:
-np.log(np.exp(-1.3863-0.439733) + np.exp(0.7493-0.031824) + \
np.exp(-0.8956-0.236589))

np.float64(-0.9292592605989974)